# Image formed by a thin lens

*optics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/thin_lens_image)

The thin-lens equation tells where the image of an object appears and how big it is. Three principal rays are traced for every object position at once, while the object walks from beyond $2f$ to inside the focal length and the real, inverted image turns into a virtual, upright one.

**Vector form**

$$\frac{1}{s} + \frac{1}{s'} = \frac{1}{f},\qquad m = \frac{h'}{h} = -\frac{s'}{s},\qquad \theta' = \theta - \frac{y}{f}$$

**Index (tensor) form**

$$s'_k = \left(\frac{1}{f} - \frac{1}{s_k}\right)^{-1},\quad m_k = -\frac{s'_k}{s_k},\quad \theta'_{kr} = \theta_{kr} - \frac{y_{kr}}{f}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $s_k$ | `s` | (K,) | object distance in animation frame $k$ |
| $s'_k$ | `image_distance(s, f)` | (K,) | image distance; negative ⇒ virtual image on the object side |
| $m_k$ | `magnification(s, s_img)` | (K,) | image height / object height; negative ⇒ inverted |
| $\theta_{kr}$ | `slope_in` | (K, 3) | slope of principal ray $r \in \{\text{parallel, centre, focal}\}$ before the lens |
| $y_{kr}$ | `height_at_lens(h, s, slope_in)` | (K, 3) | where each principal ray crosses the lens |
| $\theta'_{kr}$ | `lens_kink(y, slope_in, f)` | (K, 3) | slope after the lens: every ray is bent by $-y/f$ |

Distances are measured from the lens: the object sits at $x = -s$ and a real image at $x = +s'$. The three principal rays are (1) parallel to the axis, then through the far focal point; (2) straight through the centre; (3) through the near focal point, then parallel. They all follow from the single rule $\theta' = \theta - y/f$. Inside $f$ the rays diverge after the lens; the dashed extensions meet at the virtual image. At $s = f$ the image runs off to infinity.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/thin_lens_image/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    f, h = params["f"], params["h"]
    s = tensors.object_distances(params["s_start"], params["s_end"], f)
    slope_in = tensors.principal_slopes(h, s, f)

    result = solve.solve(s, h, f, slope_in)

    for k in range(0, len(s), 20):
        kind = "real, inverted" if result["s_img"][k] > 0 else "virtual, upright"
        print(f"s = {s[k]:5.1f} cm  ->  s' = {result['s_img'][k]:7.1f} cm,  m = {result['m'][k]:6.2f}  ({kind})")
    return plot.animate(s, h, f, result, out_dir / "thin_lens_image.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/thin_lens_image/equation.py
r"""Physics → tensor translation.

A thin lens of focal length f images an object at distance s to a distance s' behind the lens:

    \frac{1}{s} + \frac{1}{s'} = \frac{1}{f},        m = \frac{h'}{h} = -\frac{s'}{s}

s' < 0 means the image is on the same side as the object (virtual); m < 0 means it is upside down.

Behind both formulas is one rule for rays (paraxial, small angles): a ray crossing the lens at
height y keeps its height but its slope θ is bent by -y/f,

    \theta' = \theta - y / f

We treat every animation frame k (object distance s_k) and every principal ray r at once, so
the tensors carry indices k and r:

    s'_k = (1/f - 1/s_k)^{-1},    m_k = -s'_k / s_k,    \theta'_{kr} = \theta_{kr} - y_{kr} / f

No index is repeated, so nothing is summed: these are element-wise operations on tensors.
"""

import torch


def image_distance(s: torch.Tensor, f: float) -> torch.Tensor:
    """s' = 1 / (1/f − 1/s).   s: (K,) → (K,)"""
    return 1.0 / (1.0 / f - 1.0 / s)


def magnification(s: torch.Tensor, s_img: torch.Tensor) -> torch.Tensor:
    """m = −s'/s.   (K,) → (K,)"""
    return -s_img / s


def height_at_lens(h: float, s: torch.Tensor, slope: torch.Tensor) -> torch.Tensor:
    """A ray leaving the object tip (x = −s, y = h) with slope θ reaches the lens at y = h + θ s.

    s: (K,), slope: (K, R) → (K, R)
    """
    return h + slope * s[:, None]


def lens_kink(y: torch.Tensor, slope: torch.Tensor, f: float) -> torch.Tensor:
    """θ' = θ − y/f: the slope of each ray after the lens.   (K, R) → (K, R)"""
    return slope - y / f


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/thin_lens_image/tensors.py
"""Build the concrete tensors: the object distance in every frame and the slopes of the three principal rays."""

import torch


def object_distances(s_start: float, s_end: float, f: float, n_frames: int = 120) -> torch.Tensor:
    """Object distance s_k for every frame k: shape (K,).

    s = f exactly would put the image at infinity, so any sample landing on it is nudged a little.
    """
    s = torch.linspace(s_start, s_end, n_frames)
    return torch.where((s - f).abs() < 1e-3, s + 1e-2, s)


def principal_slopes(h: float, s: torch.Tensor, f: float) -> torch.Tensor:
    """Slope of each principal ray as it leaves the object tip (x = −s, y = h): shape (K, 3).

    column 0: parallel ray          θ = 0
    column 1: ray through the centre  θ = −h / s            (aims at x = 0, y = 0)
    column 2: ray through near focus  θ = −h / (s − f)      (aims at x = −f, y = 0)
    """
    parallel = torch.zeros_like(s)
    centre = -h / s
    focal = -h / (s - f)
    return torch.stack([parallel, centre, focal], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/thin_lens_image/solve.py
"""Apply the thin-lens rules to every frame: image position, magnification and the principal rays."""

import torch

from .equation import height_at_lens, image_distance, lens_kink, magnification


def solve(s: torch.Tensor, h: float, f: float, slope_in: torch.Tensor) -> dict[str, torch.Tensor]:
    """Return a dict of tensors, one row per frame k.

    s_img (K,), m (K,), image height h_img (K,),
    y (K, 3) where each principal ray crosses the lens, slope_out (K, 3) its slope afterwards.
    """
    s_img = image_distance(s, f)
    m = magnification(s, s_img)
    y = height_at_lens(h, s, slope_in)
    slope_out = lens_kink(y, slope_in, f)
    return {"s_img": s_img, "m": m, "h_img": m * h, "y": y, "slope_out": slope_out}


def height_after_lens(y: torch.Tensor, slope_out: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Height of each ray at a distance x behind the lens: y + θ' x.   (K, R), (K,) → (K, R)"""
    return y + slope_out * x[:, None]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/thin_lens_image/plot.py
"""Animate the object sliding towards the lens: principal rays, the image and its type (real / virtual)."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(s: torch.Tensor, h: float, f: float, res: dict, path: Path) -> Path:
    s = s.numpy()  # (K,)
    s_img, m, h_img = res["s_img"].numpy(), res["m"].numpy(), res["h_img"].numpy()  # (K,) each
    y, slope_out = res["y"].numpy(), res["slope_out"].numpy()  # (K, 3) each
    x_min, x_max = -1.1 * s.max(), max(3.5 * f, 0.5 * s.max())
    y_max = 3.5 * h

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-y_max, y_max)
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("height (cm)")
    ax.axhline(0, color="#6b7280", lw=0.8)
    lens_kw = {"arrowstyle": "<|-|>", "color": "#2563eb", "lw": 2, "mutation_scale": 12}
    ax.annotate("", (0, 0.95 * y_max), (0, -0.95 * y_max), arrowprops=lens_kw)
    for x, name in [(-2 * f, "2F"), (-f, "F"), (f, "F"), (2 * f, "2F")]:
        ax.plot([x], [0], "k|", ms=8)
        ax.text(x, -0.1 * h, name, ha="center", va="top", fontsize=8)

    colors = ["#dc2626", "#16a34a", "#9333ea"]  # parallel, centre, focal ray
    names = ["parallel → F", "through centre", "through F → parallel"]
    before = [ax.plot([], [], lw=1.3, color=c)[0] for c in colors]
    after = [ax.plot([], [], lw=1.3, color=c, label=n)[0] for c, n in zip(colors, names)]
    behind = [ax.plot([], [], ":", lw=1.0, color=c)[0] for c in colors]  # back-extensions to a virtual image
    arrow_kw = {"arrowstyle": "-|>", "lw": 2.5, "mutation_scale": 14}
    obj = ax.annotate("", (0, 0), (0, 0), arrowprops={**arrow_kw, "color": "#111827"})
    img = ax.annotate("", (0, 0), (0, 0), arrowprops={**arrow_kw, "color": "#ea580c"})
    ax.legend(loc="lower right", fontsize=7)
    title = ax.set_title(" ")

    def update(k: int):
        obj.xy, obj.xyann = (-s[k], h), (-s[k], 0)  # object arrow from the axis up to its tip
        img.xy, img.xyann = (s_img[k], h_img[k]), (s_img[k], 0)
        real = s_img[k] > 0
        img.arrow_patch.set_linestyle("-" if real else "--")
        for r in range(3):
            before[r].set_data([-s[k], 0], [h, y[k, r]])  # object tip → lens
            after[r].set_data([0, x_max], [y[k, r], y[k, r] + slope_out[k, r] * x_max])  # lens → right edge
            if real:
                behind[r].set_data([], [])
            else:  # continue the outgoing ray backwards to the virtual image at x = s' < 0
                behind[r].set_data([0, s_img[k]], [y[k, r], y[k, r] + slope_out[k, r] * s_img[k]])
        kind = "real, inverted" if real else "virtual, upright"
        title.set_text(f"s = {s[k]:.1f} cm   s' = {s_img[k]:.1f} cm   m = {m[k]:.2f}   ({kind})")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(s), 90), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'f': 10.0,
    'h': 3.0,
    's_start': 30.0,
    's_end': 5.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.thin_lens_image")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.thin_lens_image.main")
problem_dir = Path("problems/optics/thin_lens_image")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
